# モデルロード

In [ ]:
import torch
import os
from transformer_lens import HookedTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

# ==========================================
# 1. 環境セットアップ (Setup)
# ==========================================
os.environ["TOKENIZERS_PARALLELISM"] = "false"
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"🚀 使用デバイス: {device}")

# ==========================================
# 2. Gemmaモデル(Base)のロード
# ==========================================
# ※ここが一番時間がかかるので独立させます
print("🧠 Gemma-2-2B (Base) をロード中...")

# BaseモデルをTransformersで読み込み
# (HookedTransformerの自動ロードだとGemma2特有のエラーが出やすいため、手動ロードを経由します)
hf_model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2-2b", 
    torch_dtype=torch.bfloat16, 
    device_map=device
)
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b")

# HookedTransformerでラップ（エラー回避オプション付き）
model = HookedTransformer.from_pretrained(
    "gemma-2-2b", 
    hf_model=hf_model,
    device=device,
    tokenizer=tokenizer,
    dtype=torch.bfloat16,
    fold_ln=False,
    center_writing_weights=False,
    center_unembed=False,
    fold_value_biases=False,
)

print("✅ Baseモデル準備完了！次のセルへ進んでください。")

# SAE-Lensで観察

In [ ]:
import torch
from sae_lens import SAE

# ==========================================
# 1. 解析設定 (Configuration)
# ==========================================
# ★ここを変更するだけで、自動的に対象の層（Layer）が切り替わります★
LAYER = 15                 # 見たい層の番号 (例: 12, 20)
INPUT_TEXT = "I study Philosohie. So, "  # 解析したいテキスト

# ==========================================
# 2. 接続ポイントの自動特定 (Auto-Resolution)
# ==========================================
# 面倒なID管理を自動化（canonicalを使用することで数字のズレを吸収）
SAE_RELEASE = "gemma-scope-2b-pt-res-canonical"
SAE_ID = f"layer_{LAYER}/width_16k/canonical"
HOOK_POINT = f"blocks.{LAYER}.hook_resid_post"

print(f"📍 解析対象: 第 {LAYER} 層")
print(f"   (SAE ID: {SAE_ID})")

# ==========================================
# 3. SAEのロード (Loading)
# ==========================================
print(f"🔄 SAEをロード中...")
sae = SAE.from_pretrained(
    release=SAE_RELEASE,
    sae_id=SAE_ID,
    device=device
)[0]

# ==========================================
# 4. 特徴量解析の実行 (Execution)
# ==========================================
# (1) テキストをモデルに入力し、指定層の生のアクティベーション（脳波）を取得
_, cache = model.run_with_cache(INPUT_TEXT, prepend_bos=True)
original_act = cache[HOOK_POINT]

# (2) SAEを通して、アクティベーションを「意味のある特徴量」に分解
feature_acts = sae.encode(original_act)

# アクティブな特徴量の総数を数える
active_count = (feature_acts[0, -1, :] > 0.1).sum().item()
print(f"📊 アクティブな特徴量: {active_count} 個 / 16384 個")

# (3) トップk個を表示
top_k = 10  # 見やすい数に調整
top_values, top_indices = torch.topk(feature_acts[0, -1, :], k=top_k)

# ==========================================
# 5. 結果の表示 (Output)
# ==========================================
print(f"\n📖 入力テキスト: 「{INPUT_TEXT}」")
print(f"\n🔬 観察結果 (トップ {top_k}):")
print("-" * 80)

for i in range(top_k):
    feature_id = top_indices[i].item()
    strength = top_values[i].item()
    
    # 強度が強いものにアイコンをつける
    status = "🔥" if strength > 5.0 else "  "
    
    # Neuronpediaの短縮エイリアス形式を使用
    # 例: https://www.neuronpedia.org/gemma-2-2b/20-gemmascope-res-16k/4945
    url = f"https://www.neuronpedia.org/gemma-2-2b/{LAYER}-gemmascope-res-16k/{feature_id}"
    
    print(f"{status} Rank {i+1:<2} | ID: {feature_id:<6} | 強度: {strength:.2f}")
    print(f"   🔗 {url}")
    print("-" * 80)

# ==========================================
# 6. 続きを作成させる (Generation)
# ==========================================
print("\n--- Gemma (Base) の続き作成 ---")
# BaseモデルはInstruction Tuningされていないので、唐突に続きを書き始めます
output_text = model.generate(INPUT_TEXT, max_new_tokens=100, temperature=0.7)
print(output_text)